[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/094.%20The%20Multi-Echelon%20Inventory%20Optimization%20Problem/P94-Tier-4_executed.ipynb)

---

# 94. The Multi-Echelon Inventory Optimization Problem

## Tier 4: Demand-pattern learning with variational autoencoders

### Purpose and workflow

This notebook demonstrates how a variational autoencoder (VAE) can learn a compact probability model of demand patterns and generate synthetic demand scenarios. Those scenarios are then used to evaluate a simple inventory policy and to search a finite grid of candidate policies. The workflow is:

1. Generate a demonstration history for four locations.
2. Convert overlapping seven-day windows into demand-pattern samples.
3. Train a VAE to reconstruct those samples while regularizing its latent representation.
4. Sample the learned latent distribution and decode synthetic demand windows.
5. Evaluate candidate inventory levels on generated scenarios and select the lowest-cost grid point.
6. Compare that policy with a mean-plus-two-standard-deviations benchmark on the same evaluation scenarios.

The neural network learns a distribution of demand windows; it does not directly optimize inventory. The inventory search remains the explicit grid search shown later in the notebook.

### Scope and modeling assumptions

- The demonstration data are generated from seasonal, weekly, location-specific patterns with random noise. They are not observed business data, and no random seed is set, so numerical results can vary between runs.
- Each sample contains seven consecutive days for four locations, flattened into 28 features. Overlapping windows are dependent observations, not independent weeks.
- The VAE uses a Gaussian latent prior, a mean-squared reconstruction term, and a weighted Kullback–Leibler (KL) regularization term. Demand features are not standardized in this example.
- Decoded demand is constrained to be nonnegative when scenarios are generated. The decoder itself has a linear output layer; the nonnegativity constraint is applied after decoding.
- The inventory calculation is a simplified, separable, one-period comparison of a stock target with each location's average daily demand over a seven-day scenario. It does not model replenishment lead times, inventory flows between echelons, capacity, fixed order costs, or time-evolving stock. Consequently, it illustrates scenario-based policy evaluation rather than a complete multi-echelon inventory model.
- The finite grid search is exact only over the listed candidate levels. It does not establish a continuous or globally optimal policy outside that grid.
- Scenario generation, grid search, and policy comparison are demonstrations of how learned scenarios can enter an optimization workflow. They do not prove that a VAE improves real-world service or cost; that requires held-out data and an appropriately specified operational model.

### VAE background and mathematical model

An encoder maps an observed demand window $x$ to the parameters of a latent Gaussian distribution. A latent vector $z$ is sampled from that distribution, and a decoder maps $z$ back to a reconstructed window. The stochastic latent representation encourages the model to organize demand patterns into a continuous space from which new samples can be drawn.

For a sample $x \in \mathbb{R}^{28}$, the encoder returns a mean vector $\mu(x)$ and log-variance vector $\log \sigma^2(x)$, each in $\mathbb{R}^{8}$. The reparameterization step is

$$
z = \mu(x) + \sigma(x) \odot \epsilon,
\qquad \epsilon \sim \mathcal{N}(0, I),
\qquad \sigma(x) = \exp\!\left(\tfrac{1}{2}\log \sigma^2(x)\right).
$$

Here, $\odot$ denotes elementwise multiplication and $I$ is the identity covariance matrix. Writing the random draw as a differentiable transformation of $\epsilon$ allows gradients to pass through the sampling step during training.

The loss for a batch $B$ is

$$
\mathcal{L}(B) = \sum_{x \in B} \left[
\underbrace{\lVert x - \hat{x} \rVert_2^2}_{\text{reconstruction loss}}
+ \beta\,\underbrace{\left(-\tfrac{1}{2}\sum_{j=1}^{d_z}
\left[1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right]\right)}_{D_{\mathrm{KL}}(q(z\mid x)\,\|\,\mathcal{N}(0,I))}
\right].
$$

In this expression, $\hat{x}$ is the decoder output, $d_z=8$ is the latent dimension, and $\beta=1$ weights the KL term. The reconstruction term encourages fidelity to observed windows; the KL term encourages each encoded distribution to remain near the standard-normal prior. The code sums feature-wise squared errors and KL contributions within a sample, then averages the batch loss per sample for the optimizer step.

### Scenario-based inventory evaluation

For scenario $s$, let $d_{s\ell t}$ be the demand at location $\ell$ on day $t$, and let $p_\ell$ be the candidate stock target. The demonstration evaluates the seven-day average demand

$$
\bar d_{s\ell} = \frac{1}{7}\sum_{t=1}^{7} d_{s\ell t}
$$

using the separable overage/shortage cost

$$
C(p; s) = \sum_{\ell=1}^{4}
\left[h_\ell\max(p_\ell-\bar d_{s\ell},0)
+ b_\ell\max(\bar d_{s\ell}-p_\ell,0)\right].
$$

Here, $h_\ell$ and $b_\ell$ are the location-specific holding and backorder cost weights used in the code. The average scenario cost is the arithmetic mean of $C(p;s)$ over the generated scenarios. The policy search enumerates the Cartesian product of four ten-point candidate grids and returns the candidate with the lowest average cost. A tie can occur; the implementation keeps the first minimum encountered in the grid's iteration order.

### Inputs, outputs, and interpretation

The inputs are the generated daily demand history, the stated VAE configuration, and the candidate stock-level grids. Outputs include training-loss histories, reconstruction and scenario summaries, visualizations, and the lowest-cost candidate policy found by grid search. The results describe this demonstration data and this simplified cost model only; they are not evidence of forecast accuracy or operational savings.

## 1. Import libraries and choose the computation device

PyTorch is required for the VAE. The device selection uses a CUDA-enabled GPU when available and otherwise uses the CPU. The other imports support data handling, plotting, timing, and type annotations.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Define the VAE architecture

The configuration keeps the input size, latent size, hidden-layer widths, learning rate, batch size, epoch count, and KL weight explicit. The encoder produces two outputs because a Gaussian distribution is specified by its mean and variance. The decoder reverses the hidden-layer widths and maps a latent vector back to the input dimension. Batch normalization is used in the hidden layers.

In [ ]:
@dataclass
class VAEConfig:
    """Hyperparameters and layer sizes for the demand-pattern VAE."""

    input_dim: int
    latent_dim: int = 10
    hidden_dims: Optional[List[int]] = None
    learning_rate: float = 0.001
    batch_size: int = 32
    epochs: int = 1000
    beta: float = 1.0

    def __post_init__(self) -> None:
        if self.hidden_dims is None:
            self.hidden_dims = [512, 256, 128, 64]


class DemandDataset(Dataset):
    """Torch dataset that stores demand windows as floating-point tensors."""

    def __init__(self, demand_data: np.ndarray) -> None:
        self.data = torch.as_tensor(demand_data, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.data[idx]


class Encoder(nn.Module):
    """Map an observed demand window to latent Gaussian parameters."""

    def __init__(
        self, input_dim: int, hidden_dims: List[int], latent_dim: int
    ) -> None:
        super().__init__()
        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(previous_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                ]
            )
            previous_dim = hidden_dim

        self.hidden_layers = nn.Sequential(*layers)
        self.fc_mu = nn.Linear(previous_dim, latent_dim)
        self.fc_logvar = nn.Linear(previous_dim, latent_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        hidden = self.hidden_layers(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)


class Decoder(nn.Module):
    """Map a latent vector back to a demand-window reconstruction."""

    def __init__(
        self, latent_dim: int, hidden_dims: List[int], output_dim: int
    ) -> None:
        super().__init__()
        layers = []
        previous_dim = latent_dim

        for hidden_dim in reversed(hidden_dims):
            layers.extend(
                [
                    nn.Linear(previous_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                ]
            )
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, output_dim))
        self.hidden_layers = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.hidden_layers(z)

## 3. Implement VAE training and demand utilities

The training loop has three essential stages: compute the reconstruction and KL losses, backpropagate their weighted sum, and update the network parameters with Adam. Loss values are accumulated as detached numbers for reporting, while the tensor-valued loss is retained for gradient computation. The reported losses are averaged per training sample.

In [ ]:
class VariationalAutoencoder:
    """VAE for learning and sampling demand-window patterns."""

    def __init__(self, config: VAEConfig) -> None:
        self.config = config
        self.device = device
        self.encoder = Encoder(
            config.input_dim, config.hidden_dims, config.latent_dim
        ).to(self.device)
        self.decoder = Decoder(
            config.latent_dim, config.hidden_dims, config.input_dim
        ).to(self.device)
        parameters = list(self.encoder.parameters()) + list(
            self.decoder.parameters()
        )
        self.optimizer = optim.Adam(parameters, lr=config.learning_rate)
        self.training_history: Dict[str, List[float]] = {
            "reconstruction_loss": [],
            "kl_loss": [],
            "total_loss": [],
        }

    def reparameterize(
        self, mu: torch.Tensor, logvar: torch.Tensor
    ) -> torch.Tensor:
        """Sample from the encoded Gaussian while preserving gradients."""
        standard_deviation = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(standard_deviation)
        return mu + epsilon * standard_deviation

    def encode(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def loss_function(
        self,
        x_recon: torch.Tensor,
        x: torch.Tensor,
        mu: torch.Tensor,
        logvar: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return summed reconstruction, KL, and weighted total losses."""
        reconstruction_loss = F.mse_loss(x_recon, x, reduction="sum")
        kl_loss = -0.5 * torch.sum(
            1 + logvar - mu.pow(2) - logvar.exp()
        )
        total_loss = reconstruction_loss + self.config.beta * kl_loss
        return reconstruction_loss, kl_loss, total_loss

    def train_epoch(
        self, dataloader: DataLoader
    ) -> Tuple[float, float, float]:
        self.encoder.train()
        self.decoder.train()
        reconstruction_total = 0.0
        kl_total = 0.0
        loss_total = 0.0

        for data in dataloader:
            data = data.to(self.device)
            self.optimizer.zero_grad()
            x_recon, mu, logvar = self.forward(data)
            reconstruction_loss, kl_loss, total_loss = self.loss_function(
                x_recon, data, mu, logvar
            )

            # Summed feature losses are divided by batch size so the gradient
            # scale does not depend on how many samples are in this batch.
            batch_loss = total_loss / len(data)
            batch_loss.backward()
            self.optimizer.step()

            reconstruction_total += reconstruction_loss.detach().item()
            kl_total += kl_loss.detach().item()
            loss_total += total_loss.detach().item()

        sample_count = len(dataloader.dataset)
        return (
            reconstruction_total / sample_count,
            kl_total / sample_count,
            loss_total / sample_count,
        )

    def train(self, dataloader: DataLoader) -> Dict[str, List[float]]:
        print(f"Training the VAE for {self.config.epochs} epochs...")
        for epoch in range(self.config.epochs):
            reconstruction_loss, kl_loss, total_loss = self.train_epoch(
                dataloader
            )
            self.training_history["reconstruction_loss"].append(
                reconstruction_loss
            )
            self.training_history["kl_loss"].append(kl_loss)
            self.training_history["total_loss"].append(total_loss)

            if (epoch + 1) % 100 == 0:
                print(
                    f"Epoch {epoch + 1:4d}: "
                    f"Reconstruction={reconstruction_loss:.4f}, "
                    f"KL={kl_loss:.4f}, Total={total_loss:.4f}"
                )
        return self.training_history

    def generate_synthetic_demand(self, num_samples: int = 100) -> np.ndarray:
        """Draw latent samples, decode them, and enforce nonnegative demand."""
        self.encoder.eval()
        self.decoder.eval()
        with torch.no_grad():
            z = torch.randn(num_samples, self.config.latent_dim).to(self.device)
            generated = self.decode(z)
            generated = torch.clamp(generated, min=0.0)
        return generated.cpu().numpy()

    def encode_to_latent(self, demand_data: np.ndarray) -> np.ndarray:
        """Return encoded means as deterministic latent representations."""
        self.encoder.eval()
        with torch.no_grad():
            data = torch.as_tensor(
                demand_data, dtype=torch.float32, device=self.device
            )
            mu, _ = self.encode(data)
        return mu.cpu().numpy()

    def reconstruct_demand(self, demand_data: np.ndarray) -> np.ndarray:
        """Decode the latent mean for a deterministic reconstruction."""
        self.encoder.eval()
        self.decoder.eval()
        with torch.no_grad():
            data = torch.as_tensor(
                demand_data, dtype=torch.float32, device=self.device
            )
            mu, _ = self.encode(data)
            reconstruction = self.decode(mu)
        return reconstruction.cpu().numpy()

## 4. Generate demonstration demand history

Each row of the returned array represents one day, and each column represents one location. The seasonal and weekly factors create structured variation; independent noise adds day-to-day variability. The four base-demand values define the locations used throughout this example.

In [ ]:
def generate_historical_demand_data(
    num_days: int = 365, num_locations: int = 4
) -> np.ndarray:
    """Create nonnegative synthetic daily demand for four locations."""
    base_demands = np.array([20.0, 30.0, 50.0, 40.0])
    if num_locations != len(base_demands):
        raise ValueError("This demonstration defines exactly four locations.")

    demand_data = np.zeros((num_days, num_locations), dtype=float)
    for day in range(num_days):
        seasonal_factor = 1 + 0.3 * np.sin(2 * np.pi * day / 365)
        weekly_factor = 1 + 0.2 * np.sin(2 * np.pi * (day % 7) / 7)
        noise = np.random.normal(0, 0.1, size=num_locations)
        daily_demand = base_demands * seasonal_factor * weekly_factor * (1 + noise)
        demand_data[day] = np.maximum(daily_demand, 0)
    return demand_data


print("Generating demonstration demand history...")
historical_demand = generate_historical_demand_data(
    num_days=365, num_locations=4
)
print(f"Historical demand shape: {historical_demand.shape}")
print("Daily demand statistics by location:")
print(f"  Mean: {np.mean(historical_demand, axis=0)}")
print(f"  Standard deviation: {np.std(historical_demand, axis=0)}")
print(f"  Minimum: {np.min(historical_demand, axis=0)}")
print(f"  Maximum: {np.max(historical_demand, axis=0)}")

## 5. Create seven-day demand windows and configure the VAE

A sliding window preserves the local sequence of days: window $i$ contains days $i$ through $i+6$. Flattening the window gives one feature vector with $7 \times 4=28$ entries. The windows overlap, so adjacent samples share observations. The smaller network and training settings below are the demonstration configuration for these 28-feature samples.

In [ ]:
window_size = 7
num_locations = historical_demand.shape[1]
samples = np.stack(
    [
        historical_demand[start : start + window_size].reshape(-1)
        for start in range(len(historical_demand) - window_size + 1)
    ]
)

print(f"Number of seven-day windows: {len(samples)}")
print(f"Features per window: {samples.shape[1]}")

config = VAEConfig(
    input_dim=samples.shape[1],
    latent_dim=8,
    hidden_dims=[64, 32, 16],
    learning_rate=0.001,
    batch_size=16,
    epochs=500,
    beta=1.0,
)

print("\nVAE configuration:")
print(f"  Input dimension: {config.input_dim}")
print(f"  Latent dimension: {config.latent_dim}")
print(f"  Hidden dimensions: {config.hidden_dims}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Batch size: {config.batch_size}")
print(f"  Epochs: {config.epochs}")
print(f"  KL weight (beta): {config.beta}")

## 6. Train the VAE

The data loader forms shuffled mini-batches for gradient-based learning. The model parameters are updated using Adam, while the history records per-sample reconstruction, KL, and total losses after each epoch. Training loss measures fit to the training windows; it is not a held-out measure of generalization.

In [ ]:
dataset = DemandDataset(samples)
dataloader = DataLoader(
    dataset, batch_size=config.batch_size, shuffle=True
)

vae = VariationalAutoencoder(config)
start_time = time.time()
training_history = vae.train(dataloader)
training_time = time.time() - start_time

print(f"\nTraining elapsed time: {training_time:.2f} seconds")
print(
    "Final per-sample reconstruction loss: "
    f"{training_history['reconstruction_loss'][-1]:.4f}"
)
print(f"Final per-sample KL loss: {training_history['kl_loss'][-1]:.4f}")
print(f"Final per-sample total loss: {training_history['total_loss'][-1]:.4f}")

## 7. Evaluate reconstructions and generate scenarios

The reconstruction diagnostic compares ten observed windows with deterministic reconstructions obtained by decoding their latent means. The scenario generator instead samples from the standard-normal latent prior, decodes those draws, and clips negative outputs to zero. Summary statistics and the mean absolute differences in feature-wise means and standard deviations describe the generated sample; they do not establish that the full joint distribution is realistic.

In [ ]:
print("=== VAE evaluation ===")
test_samples = samples[:10]
reconstructed = vae.reconstruct_demand(test_samples)
mse_errors = np.mean((test_samples - reconstructed) ** 2, axis=1)

print(f"Mean reconstruction MSE: {np.mean(mse_errors):.4f}")
print(f"Standard deviation of reconstruction MSE: {np.std(mse_errors):.4f}")

print("\nGenerating synthetic demand windows...")
synthetic_scenarios = vae.generate_synthetic_demand(num_samples=100)
print(f"Generated scenarios: {len(synthetic_scenarios)}")
print(f"Synthetic feature means: {np.mean(synthetic_scenarios, axis=0)}")
print(
    "Synthetic feature standard deviations: "
    f"{np.std(synthetic_scenarios, axis=0)}"
)
print(f"Synthetic minimum: {np.min(synthetic_scenarios):.4f}")
print(f"Synthetic maximum: {np.max(synthetic_scenarios):.4f}")

original_mean = np.mean(samples, axis=0)
original_std = np.std(samples, axis=0)
synthetic_mean = np.mean(synthetic_scenarios, axis=0)
synthetic_std = np.std(synthetic_scenarios, axis=0)
print("\nOriginal-versus-synthetic feature summaries:")
print(
    "Mean absolute difference in feature means: "
    f"{np.mean(np.abs(original_mean - synthetic_mean)):.4f}"
)
print(
    "Mean absolute difference in feature standard deviations: "
    f"{np.mean(np.abs(original_std - synthetic_std)):.4f}"
)

## 8. Inspect training, reconstruction, and demand-pattern summaries

The first two latent coordinates provide a two-dimensional projection of the eight-dimensional encoded means, not a complete representation of the latent space. The plots below are diagnostic views: they help inspect loss behavior, one location's reconstruction, generated examples, marginal demand values, and reconstruction errors.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("VAE demand-pattern diagnostics", fontsize=16, fontweight="bold")

epochs = np.arange(1, len(training_history["total_loss"]) + 1)
axes[0, 0].plot(
    epochs, training_history["reconstruction_loss"], label="Reconstruction"
)
axes[0, 0].plot(epochs, training_history["kl_loss"], label="KL divergence")
axes[0, 0].plot(
    epochs, training_history["total_loss"], label="Total", linewidth=2
)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss per sample")
axes[0, 0].set_title("Training losses")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

original_reshaped = test_samples[0].reshape(window_size, num_locations)
reconstructed_reshaped = reconstructed[0].reshape(
    window_size, num_locations
)
days = np.arange(1, window_size + 1)
axes[0, 1].plot(
    days, original_reshaped[:, 0], "o-", label="Observed", linewidth=2
)
axes[0, 1].plot(
    days,
    reconstructed_reshaped[:, 0],
    "s--",
    label="Reconstructed",
    linewidth=2,
)
axes[0, 1].set_xlabel("Day within window")
axes[0, 1].set_ylabel("Demand (units per day)")
axes[0, 1].set_title("Location 1: observed and reconstructed")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

latent_codes = vae.encode_to_latent(samples[:100])
scatter = axes[0, 2].scatter(
    latent_codes[:, 0],
    latent_codes[:, 1],
    c=np.arange(len(latent_codes)),
    cmap="viridis",
    alpha=0.7,
)
axes[0, 2].set_xlabel("Latent dimension 1")
axes[0, 2].set_ylabel("Latent dimension 2")
axes[0, 2].set_title("Encoded means: first two dimensions")
fig.colorbar(scatter, ax=axes[0, 2], label="Window index")
axes[0, 2].grid(True, alpha=0.3)

for scenario_index in range(3):
    scenario = synthetic_scenarios[scenario_index].reshape(
        window_size, num_locations
    )
    axes[1, 0].plot(
        days,
        scenario[:, 0],
        marker="o",
        alpha=0.8,
        label=f"Scenario {scenario_index + 1}",
    )
axes[1, 0].set_xlabel("Day within window")
axes[1, 0].set_ylabel("Demand (units per day)")
axes[1, 0].set_title("Generated scenarios: location 1")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

original_location_values = samples[:, 0::num_locations].ravel()
synthetic_location_values = synthetic_scenarios[:, 0::num_locations].ravel()
axes[1, 1].hist(
    original_location_values,
    bins=30,
    alpha=0.55,
    density=True,
    label="Observed windows",
)
axes[1, 1].hist(
    synthetic_location_values,
    bins=30,
    alpha=0.55,
    density=True,
    label="Generated scenarios",
)
axes[1, 1].set_xlabel("Demand (units per day)")
axes[1, 1].set_ylabel("Density")
axes[1, 1].set_title("Location 1: marginal distribution")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].hist(mse_errors, bins=10, color="darkorange", edgecolor="black")
axes[1, 2].axvline(
    np.mean(mse_errors),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(mse_errors):.4f}",
)
axes[1, 2].set_xlabel("Reconstruction MSE")
axes[1, 2].set_ylabel("Number of windows")
axes[1, 2].set_title("Reconstruction-error distribution")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Evaluate policies and search the candidate grid

The optimizer first generates a fixed set of VAE scenarios. For a candidate policy, it computes each scenario's seven-day average demand at every location, then applies the overage/shortage cost defined above. The search evaluates all combinations in the four specified candidate ranges and retains the first candidate attaining the smallest average cost. The returned scenarios are also used to evaluate the benchmark policy, making the two policy costs directly comparable on the same generated sample.

In [ ]:
class VAEEnhancedOptimizer:
    """Grid-search policy evaluation using VAE-generated scenarios."""

    def __init__(
        self, vae: VariationalAutoencoder, location_names: List[str]
    ) -> None:
        if len(location_names) != 4:
            raise ValueError("The demonstration optimizer requires four locations.")
        self.vae = vae
        self.location_names = location_names
        self.holding_costs = np.array([2.0, 2.0, 1.5, 1.0])
        self.backorder_costs = np.array([10.0, 10.0, 5.0, 2.0])

    def evaluate_policy_with_scenarios(
        self, policy: np.ndarray, scenarios: np.ndarray
    ) -> float:
        """Compute mean separable overage/shortage cost across scenarios."""
        expected_features = window_size * len(self.location_names)
        if scenarios.ndim != 2 or scenarios.shape[1] != expected_features:
            raise ValueError(
                f"Each scenario must contain {expected_features} features."
            )
        if policy.shape != (len(self.location_names),):
            raise ValueError("The policy must specify one level per location.")
        if len(scenarios) == 0:
            raise ValueError("At least one scenario is required for evaluation.")

        scenario_demand = scenarios.reshape(
            -1, window_size, len(self.location_names)
        ).mean(axis=1)
        overage = np.maximum(policy[None, :] - scenario_demand, 0.0)
        shortage = np.maximum(scenario_demand - policy[None, :], 0.0)
        scenario_costs = np.sum(
            overage * self.holding_costs[None, :]
            + shortage * self.backorder_costs[None, :],
            axis=1,
        )
        return float(np.mean(scenario_costs))

    def optimize_with_vae_scenarios(
        self, num_scenarios: int = 50
    ) -> Dict[str, object]:
        """Select the lowest-cost policy among the finite grid candidates."""
        synthetic_scenarios = self.vae.generate_synthetic_demand(num_scenarios)
        search_ranges = [
            np.linspace(10, 50, 10),
            np.linspace(15, 60, 10),
            np.linspace(25, 80, 10),
            np.linspace(20, 70, 10),
        ]

        best_policy = None
        best_cost = float("inf")
        for level_1 in search_ranges[0]:
            for level_2 in search_ranges[1]:
                for level_3 in search_ranges[2]:
                    for level_4 in search_ranges[3]:
                        policy = np.array(
                            [level_1, level_2, level_3, level_4]
                        )
                        cost = self.evaluate_policy_with_scenarios(
                            policy, synthetic_scenarios
                        )
                        if cost < best_cost:
                            best_cost = cost
                            best_policy = policy.copy()

        if best_policy is None:
            raise RuntimeError("The candidate grid did not contain a policy.")
        return {
            "best_policy": best_policy,
            "best_cost": best_cost,
            "scenarios_used": num_scenarios,
            "scenarios": synthetic_scenarios,
        }


location_names = ["Retail_A", "Retail_B", "DC1", "Central_Warehouse"]
vae_optimizer = VAEEnhancedOptimizer(vae, location_names)
optimization_results = vae_optimizer.optimize_with_vae_scenarios(
    num_scenarios=100
)

print("=== VAE-scenario grid-search results ===")
print(f"Lowest average cost on the candidate grid: ${optimization_results['best_cost']:.2f}")
print(f"Scenarios used: {optimization_results['scenarios_used']}")
print("Selected candidate policy:")
for name, stock_level in zip(
    location_names, optimization_results["best_policy"]
):
    print(f"  {name}: {stock_level:.1f} demand units per day")

## 10. Compare with a statistical benchmark

The benchmark sets each location's target to the mean plus two standard deviations of its historical seven-day average demand. This is a simple statistical rule, not an optimized service-level guarantee; the familiar 95% interpretation requires distributional assumptions that are not established here. Both the benchmark and grid-search policy are evaluated using the same VAE-generated scenarios and the same cost function.

In [ ]:
def traditional_statistical_optimization(
    historical_demand: np.ndarray,
    location_names: List[str],
    evaluation_scenarios: np.ndarray,
    optimizer: VAEEnhancedOptimizer,
    window_size: int = 7,
) -> Dict[str, object]:
    """Build a mean-plus-two-standard-deviations benchmark policy."""
    if historical_demand.ndim != 2:
        raise ValueError("Historical demand must be a days-by-locations array.")
    if historical_demand.shape[1] != len(location_names):
        raise ValueError("Location names must match the demand-data columns.")
    if len(historical_demand) < window_size:
        raise ValueError("The history must contain at least one full window.")

    historical_window_averages = np.stack(
        [
            historical_demand[start : start + window_size].mean(axis=0)
            for start in range(len(historical_demand) - window_size + 1)
        ]
    )
    average_demand = historical_window_averages.mean(axis=0)
    demand_standard_deviation = historical_window_averages.std(axis=0)
    policy = average_demand + 2 * demand_standard_deviation
    cost = optimizer.evaluate_policy_with_scenarios(
        policy, evaluation_scenarios
    )
    return {"policy": policy, "cost": cost}


evaluation_scenarios = optimization_results["scenarios"]
traditional_results = traditional_statistical_optimization(
    historical_demand,
    location_names,
    evaluation_scenarios,
    vae_optimizer,
    window_size=window_size,
)

vae_cost = optimization_results["best_cost"]
traditional_cost = traditional_results["cost"]
print("=== Policy comparison on shared generated scenarios ===")
print(f"Grid-search policy cost: ${vae_cost:.2f}")
print(f"Statistical benchmark cost: ${traditional_cost:.2f}")
if traditional_cost > 0:
    relative_cost_difference = (
        traditional_cost - vae_cost
    ) / traditional_cost * 100
    print(
        "Grid-search cost difference relative to the benchmark: "
        f"{relative_cost_difference:.1f}%"
    )
else:
    print("Relative cost difference is undefined because benchmark cost is zero.")

print("\nPolicy levels:")
print(f"{'Location':<20} {'Grid search':>14} {'Benchmark':>14} {'Difference':>14}")
print("-" * 66)
for index, name in enumerate(location_names):
    grid_value = optimization_results["best_policy"][index]
    benchmark_value = traditional_results["policy"][index]
    print(
        f"{name:<20} {grid_value:>14.1f} "
        f"{benchmark_value:>14.1f} {grid_value - benchmark_value:>14.1f}"
    )

## 11. Visualize policy and scenario comparisons

These panels compare costs and policy levels, show feature-wise variability in the observed and generated windows, and display the final segment of the training-loss history. A lower estimated cost here means a lower average under the stated cost function and the shared generated scenarios; it does not by itself establish better performance on future real demand.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Scenario-based inventory policy comparison", fontsize=16)

methods = ["Grid-search policy", "Statistical benchmark"]
costs = [vae_cost, traditional_cost]
bars = axes[0, 0].bar(methods, costs, color=["#2E86AB", "#A23B72"])
axes[0, 0].set_ylabel("Average modeled cost ($)")
axes[0, 0].set_title("Cost on shared generated scenarios")
axes[0, 0].tick_params(axis="x", rotation=15)
axes[0, 0].grid(True, axis="y", alpha=0.3)
for bar, cost in zip(bars, costs):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${cost:.2f}",
        ha="center",
        va="bottom",
    )

positions = np.arange(len(location_names))
width = 0.35
axes[0, 1].bar(
    positions - width / 2,
    optimization_results["best_policy"],
    width,
    label="Grid search",
    color="#2E86AB",
)
axes[0, 1].bar(
    positions + width / 2,
    traditional_results["policy"],
    width,
    label="Statistical benchmark",
    color="#A23B72",
)
axes[0, 1].set_xlabel("Location")
axes[0, 1].set_ylabel("Target (demand units per day)")
axes[0, 1].set_title("Policy levels by location")
axes[0, 1].set_xticks(positions)
axes[0, 1].set_xticklabels(location_names, rotation=30, ha="right")
axes[0, 1].legend()
axes[0, 1].grid(True, axis="y", alpha=0.3)

scenario_diversity = np.std(evaluation_scenarios, axis=0)
original_diversity = np.std(samples, axis=0)
feature_positions = np.arange(len(original_diversity))
axes[1, 0].bar(
    feature_positions - width / 2,
    original_diversity,
    width,
    label="Observed windows",
    color="#F18F01",
)
axes[1, 0].bar(
    feature_positions + width / 2,
    scenario_diversity,
    width,
    label="Generated scenarios",
    color="#C73E1D",
)
axes[1, 0].set_xlabel("Flattened feature index")
axes[1, 0].set_ylabel("Standard deviation (demand units per day)")
axes[1, 0].set_title("Feature-wise demand variability")
axes[1, 0].legend()
axes[1, 0].grid(True, axis="y", alpha=0.3)

history_length = len(training_history["total_loss"])
final_start = max(0, history_length - 100)
final_epochs = np.arange(final_start + 1, history_length + 1)
axes[1, 1].plot(
    final_epochs,
    training_history["total_loss"][final_start:],
    color="green",
    linewidth=2,
)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Total loss per sample")
axes[1, 1].set_title("Recent training-loss history")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Read the results cautiously

The printed losses summarize training on the generated overlapping windows. Reconstruction MSE measures average squared feature error for the selected test windows, which are drawn from the same synthetic history rather than a held-out period. Generated-scenario summaries and plots provide exploratory checks of marginal values and variability, but do not verify temporal dependence, cross-location dependence, calibration, or out-of-sample realism.

The grid-search policy is the lowest-cost candidate among the specified finite combinations under the sampled scenarios and simplified separable cost model. The benchmark comparison uses that same scenario set, which controls for scenario-sample differences in this particular comparison. It does not account for uncertainty in the estimated costs or establish that learned scenarios are more representative than the historical data.

A practical application would require suitable observed data, a validation design that respects time order, and an operational inventory model that represents the relevant replenishment and network constraints. Those elements are outside this demonstration.